In [6]:
import os
import sys
from datasets import load_dataset
from dotenv import load_dotenv
from datetime import datetime
sys.path.append(os.path.abspath("../.."))
from transformation.common.spark_utils import get_layer_path
load_dotenv()

True

In [7]:
endpoint_url = os.getenv("ENDPOINT_URL")
key = os.getenv("MINIO_ROOT_USER")
secret = os.getenv("MINIO_ROOT_PASSWORD")

In [8]:
def load_data_for_nlp(s3a_path):
    print(f"[*] Đang kết nối tới MinIO tại {endpoint_url}...")
    print(f"[*] Nạp dữ liệu từ: {s3a_path}")
    try:
        dataset = load_dataset(
            "parquet",
            data_files=s3a_path,
            storage_options={
                "key": key,
                "secret": secret,
                "client_kwargs":{
                    "endpoint_url": endpoint_url
                }
            }
        )
        hf_dataset = dataset["train"]
        print(f"Đã nạp xong Dataset!")
        print(f"Tổng số bản ghi {len(hf_dataset)}")
        print(f"Các cột dữ liệu: {hf_dataset.column_names}")
        
        return hf_dataset
    except Exception as e:
        print(f"[!] Lỗi khi nạp dữ liệu trực tiếp: {e}")
        return None

In [9]:
# path = os.path.abspath(os.path.join(os.path.dirname(__file__), "../.."))
# print(path)
bucket_name = "silver"
base_prefix = "nlp/dataset"
s3_path = get_layer_path("s3://", bucket_name, base_prefix) 
print(s3_path)
dataset = load_data_for_nlp(s3_path + "*.parquet")
if dataset:
    # Kiểm tra thử 1 dòng để xem cấu trúc
    print("\n[Mẫu dữ liệu dòng 0]:")
    print(dataset[0])


s3://silver/nlp/dataset/2026/03/30/
[*] Đang kết nối tới MinIO tại http://localhost:9000...
[*] Nạp dữ liệu từ: s3://silver/nlp/dataset/2026/03/30/*.parquet
Đã nạp xong Dataset!
Tổng số bản ghi 70078
Các cột dữ liệu: ['review_id', 'imdb_id', 'content', 'rating', 'source_system']

[Mẫu dữ liệu dòng 0]:
{'review_id': '000524abf6de59d91f3e4a4375ec8415cc5a9cdf1fe595fe334662b4f298d362', 'imdb_id': 'tt27543632', 'content': "For the first hour of the movie, I honestly didn't even know why I was there. It felt cringey, overly sweet, and like I was a 12-year-old getting excited over BookTok edits. But then everything changed. The story completely flipped, the tone turned tense, dark, and slightly psychological. And then came the insane plot twist which was simply wow.\n\nLooking back, the overly innocent and simple beginning was clearly intentional, and that's exactly what made the twist hit so hard. It felt like it came out of nowhere, in the best possible way.\n\nGreat casting, strong perform

In [26]:
print(type(dataset))
for i in range(5):
    for key, values in dataset[i].items():
        print(f"{key}: {values}")
    print("\n")

<class 'datasets.arrow_dataset.Dataset'>
review_id: 000524abf6de59d91f3e4a4375ec8415cc5a9cdf1fe595fe334662b4f298d362
imdb_id: tt27543632
content: For the first hour of the movie, I honestly didn't even know why I was there. It felt cringey, overly sweet, and like I was a 12-year-old getting excited over BookTok edits. But then everything changed. The story completely flipped, the tone turned tense, dark, and slightly psychological. And then came the insane plot twist which was simply wow.

Looking back, the overly innocent and simple beginning was clearly intentional, and that's exactly what made the twist hit so hard. It felt like it came out of nowhere, in the best possible way.

Great casting, strong performances, beautiful cinematography, and a solid soundtrack. The writing and story are sharp and well-crafted.

It's only the beginning of the year, but I already believe this might be the movie of 2026.
rating: 7.0
source_system: imdb


review_id: 000a465558f22ae39464a9830ce91dbcd77